# 20 — Download continuous waveforms (daily), resample/merge (cache-aware)

Writes one daily miniSEED per station containing Z channels.

In [ ]:
%run 00_config.ipynb
from obspy import read_inventory, read

In [ ]:
# Load inventory
stationxml_path = os.path.join(ROOT, "Redoubt_0p2deg_StationXML.xml")
assert file_exists(stationxml_path), "Run 10_stationxml_and_events.ipynb first"
inv = read_inventory(stationxml_path)

# Choose channel families; vertical only
keep_prefixes = ("EH", "BH", "HH", "SH")

bulk_ids = []
for net in inv:
    for sta in net:
        for cha in sta:
            if not cha.code.startswith(keep_prefixes):
                continue
            if not cha.code.endswith("Z"):
                continue
            loc = cha.location_code or ""
            bulk_ids.append((net.code, sta.code, loc, cha.code))

bulk_ids = sorted(set(bulk_ids))
print("Z-channel requests:", len(bulk_ids))
bulk_ids[:12]


In [ ]:
def harmonize_resample_merge(st: Stream, day: UTCDateTime, day_end: UTCDateTime) -> Stream:
    # For each SEED id, resample all segments to a common sampling rate (max SR seen that day),
    # then merge and trim to exact [day, day_end). Gaps are padded with NaN.
    st = st.copy()
    st.sort(keys=["network", "station", "location", "channel", "starttime"])
    st = st.split()

    out = Stream()
    for tr_id in sorted(set(tr.id for tr in st)):
        sts = st.select(id=tr_id).copy()
        srs = sorted(set(float(tr.stats.sampling_rate) for tr in sts))
        target_sr = max(srs)

        for tr in sts:
            if float(tr.stats.sampling_rate) != target_sr:
                tr.interpolate(sampling_rate=target_sr, method="linear", starttime=tr.stats.starttime)

        sts.merge(method=1, fill_value=np.nan)
        sts.trim(day, day_end, pad=True, fill_value=np.nan)

        out += sts

    out.sort(keys=["network", "station", "location", "channel", "starttime"])
    return out


In [ ]:
wf_dir = ensure_dir(os.path.join(ROOT, "waveforms_daily"))

for day in list_daily_dates(t0, t1):
    day_end = day + 24 * 3600

    # Check if we already have outputs for this day (any station file)
    already = any(fn.endswith(f".{day.date}.Z.mseed") for fn in os.listdir(wf_dir))
    if already:
        print("Day", day.date, "already has outputs in", wf_dir, "- skipping download.")
        continue

    print("Downloading waveforms for", day.date)
    st = EARTHSCOPE.get_waveforms_bulk(
        bulk=[(n, s, l, c, day, day_end) for (n, s, l, c) in bulk_ids],
        attach_response=False,
    )

    st2 = harmonize_resample_merge(st, day, day_end)

    dup = summarize_duplicates(st2)
    if dup:
        print("WARNING: still duplicated IDs after resample/merge:", dup[:10])

    for (net, sta) in sorted({(tr.stats.network, tr.stats.station) for tr in st2}):
        st_sta = st2.select(network=net, station=sta)
        if not st_sta:
            continue
        fname = f"{net}.{sta}.{day.date}.Z.mseed"
        st_sta.write(os.path.join(wf_dir, fname), format="MSEED")

    print("Wrote day", day.date, "files:", len({(tr.stats.network, tr.stats.station) for tr in st2}))
